# ORB-SLAM3 Project Demo

This notebook summarizes the ORB-SLAM3 portion of the monocular localization / mapping semester project.

ORB-SLAM3 is used here as a classical feature-based monocular SLAM baseline for comparison against the DeepVO visual odometry workflow.

## Project Context

The project uses two custom monocular video sequences that were already converted into extracted frame folders:

- `data/custom/extracted_frames/indoor_loop/`
- `data/custom/extracted_frames/outdoor_loop/`

ORB-SLAM3 was run on these frame folders using a lightweight KITTI-style adapter script. The resulting trajectories were saved in TUM trajectory format.

## macOS Setup Note

ORB-SLAM3 was built successfully on a MacBook M4 after applying minimal Apple Silicon/macOS compatibility fixes.

A key macOS issue was the Pangolin viewer. The viewer crashed with a main-thread error:

```text
nextEventMatchingMask should only be called from the Main Thread!
```

Because the crash came from the Pangolin viewer thread, the monocular workflow was run in headless mode using:

```bash
--no-viewer
```

This keeps trajectory saving enabled while avoiding the macOS viewer crash.

## Important Project Paths

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
ORB_ROOT = PROJECT_ROOT / 'src' / 'models' / 'orb_slam3'

paths = {
    'ORB-SLAM3 root': ORB_ROOT,
    'Custom settings': PROJECT_ROOT / 'configs' / 'orbslam3_custom_1280x720.yaml',
    'Indoor trajectory': PROJECT_ROOT / 'outputs' / 'trajectories' / 'orb_slam3' / 'indoor_loop' / 'keyframe_trajectory_tum.txt',
    'Outdoor trajectory': PROJECT_ROOT / 'outputs' / 'trajectories' / 'orb_slam3' / 'outdoor_loop' / 'keyframe_trajectory_tum.txt',
    'Indoor plot': PROJECT_ROOT / 'outputs' / 'plots' / 'orb_slam3' / 'indoor_loop_trajectory.png',
    'Outdoor plot': PROJECT_ROOT / 'outputs' / 'plots' / 'orb_slam3' / 'outdoor_loop_trajectory.png',
}

for label, path in paths.items():
    print(f'{label}: {path} | exists={path.exists()}')

ORB-SLAM3 root: /Users/ray/monocular-slam-project/monocular_slam_project/src/models/orb_slam3 | exists=True
Custom settings: /Users/ray/monocular-slam-project/monocular_slam_project/configs/orbslam3_custom_1280x720.yaml | exists=True
Indoor trajectory: /Users/ray/monocular-slam-project/monocular_slam_project/outputs/trajectories/orb_slam3/indoor_loop/keyframe_trajectory_tum.txt | exists=True
Outdoor trajectory: /Users/ray/monocular-slam-project/monocular_slam_project/outputs/trajectories/orb_slam3/outdoor_loop/keyframe_trajectory_tum.txt | exists=True
Indoor plot: /Users/ray/monocular-slam-project/monocular_slam_project/outputs/plots/orb_slam3/indoor_loop_trajectory.png | exists=True
Outdoor plot: /Users/ray/monocular-slam-project/monocular_slam_project/outputs/plots/orb_slam3/outdoor_loop_trajectory.png | exists=True


## Running ORB-SLAM3 Headless

The custom runner creates a KITTI-style temporary input layout with:

```text
kitti_input/
├── image_0/
│   ├── 000000.png
│   ├── 000001.png
│   └── ...
└── times.txt
```

The runner uses symlinks, so it does not duplicate the extracted frame images.

In [2]:
# Indoor run command
print('python scripts/inference/run_orbslam3_kitti_custom.py --sequence indoor_loop --fps 30 --no-viewer')

# Outdoor run command
print('python scripts/inference/run_orbslam3_kitti_custom.py --sequence outdoor_loop --fps 30 --no-viewer')

python scripts/inference/run_orbslam3_kitti_custom.py --sequence indoor_loop --fps 30 --no-viewer
python scripts/inference/run_orbslam3_kitti_custom.py --sequence outdoor_loop --fps 30 --no-viewer


## Trajectory Format

ORB-SLAM3 saves keyframe trajectories in TUM format:

```text
timestamp tx ty tz qx qy qz qw
```

For plotting, this project uses a 2D top-down view with `tx` on the horizontal axis and `tz` on the vertical axis.

In [3]:
def count_poses(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip() and not line.startswith('#'))

indoor_traj = paths['Indoor trajectory']
outdoor_traj = paths['Outdoor trajectory']

print(f'Indoor keyframe poses: {count_poses(indoor_traj)}')
print(f'Outdoor keyframe poses: {count_poses(outdoor_traj)}')

Indoor keyframe poses: 523
Outdoor keyframe poses: 182


## Indoor Custom Sequence Result

The indoor sequence produced a usable trajectory with **523 keyframe poses**.

The plot is saved at:

```text
outputs/plots/orb_slam3/indoor_loop_trajectory.png
```

![Indoor ORB-SLAM3 trajectory](../outputs/plots/orb_slam3/indoor_loop_trajectory.png)

## Outdoor Custom Sequence Result

The outdoor sequence initially produced an empty trajectory when using KITTI settings. After switching to the custom 1280x720 settings file, ORB-SLAM3 produced a usable trajectory with **182 keyframe poses**.

The plot is saved at:

```text
outputs/plots/orb_slam3/outdoor_loop_trajectory.png
```

![Outdoor ORB-SLAM3 trajectory](../outputs/plots/orb_slam3/outdoor_loop_trajectory.png)

## Plotting Commands

The ORB-SLAM3 plotter reads TUM files and writes PNG plots under `outputs/plots/orb_slam3/`.

In [4]:
print('.venv/bin/python scripts/visualization/plot_orbslam3_trajectory.py outputs/trajectories/orb_slam3/indoor_loop/keyframe_trajectory_tum.txt')
print('.venv/bin/python scripts/visualization/plot_orbslam3_trajectory.py outputs/trajectories/orb_slam3/outdoor_loop/keyframe_trajectory_tum.txt')

.venv/bin/python scripts/visualization/plot_orbslam3_trajectory.py outputs/trajectories/orb_slam3/indoor_loop/keyframe_trajectory_tum.txt
.venv/bin/python scripts/visualization/plot_orbslam3_trajectory.py outputs/trajectories/orb_slam3/outdoor_loop/keyframe_trajectory_tum.txt


## Discussion

The ORB-SLAM3 workflow provides a useful feature-based monocular SLAM baseline for comparison against DeepVO.

Observed strengths:

- Produces standard TUM trajectory outputs.
- Can run on the same custom frame sequences used by DeepVO.
- Headless mode makes the workflow practical on macOS.
- Indoor custom tracking produced a clearer trajectory than the outdoor run.

Observed limitations:

- Pangolin viewer mode was not reliable on macOS, so headless mode was required.
- The custom camera settings are approximate and not a true calibration.
- Monocular SLAM has arbitrary scale without external scale information.
- The outdoor result is shorter and less stable, likely due to motion, lighting, texture, and calibration quality.

## Summary

ORB-SLAM3 was successfully integrated as the classical monocular SLAM baseline. Both indoor and outdoor custom sequences produced TUM-format trajectory outputs and saved trajectory plots. The outdoor result is usable but less stable than the indoor result, which is important to mention honestly in the final project report.

## Final Project Update: outdoor_loop2 and Headless Replay

`outdoor_loop2` was added as a longer portrait-oriented custom sequence. ORB-SLAM3 was run with a sequence-specific approximate settings file at `configs/orbslam3_custom_outdoor_loop2_1280x2276.yaml`.

ORB-SLAM3 saved a valid TUM trajectory for `outdoor_loop2`, but the run was less stable than the indoor sequence. The log contains repeated local-map tracking failures and map resets, so this result is best discussed as a robustness/stress-test case rather than a clean trajectory.

The ORB-SLAM3 pseudo-real-time demo remains replay-based. For difficult sequences, `--sync-mode even` gives smoother presentation playback because saved keyframes can be sparse or delayed.


In [ ]:
# ORB-SLAM3 outdoor_loop2 replay command
!../.venv/bin/python ../scripts/visualization/orbslam3_realtime_demo.py \
  --frames ../data/custom/extracted_frames/outdoor_loop2 \
  --trajectory ../outputs/trajectories/orb_slam3/outdoor_loop2/keyframe_trajectory_tum.txt \
  --fps 24 \
  --sync-mode even
